# Stage 2 — Information Extraction

Runs IE on the **latest admission note** per patient. Prior admissions are passed as **history**.
Uses the diagnosis-redacted note from Stage 1.5 (not the raw cohort note) so the diagnosis stated
in the discharge summary itself can't leak into extraction.

**LLM:** `LLM_PROVIDER` in `pipeline_config.py` (`openrouter` | `ollama`)

**Input:** `cohort.pkl` (Stage 1) + redacted notes (Stage 1.5) | **Checkpoint:** `ie_checkpoint.json`

**Next:** `stage_03_symptom_tree.ipynb`


## Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / "pipeline.py").exists():
    NB_DIR = ROOT
elif (ROOT / "notebooks" / "pipeline.py").exists():
    NB_DIR = ROOT / "notebooks"
else:
    NB_DIR = ROOT.parent / "notebooks"
sys.path.insert(0, str(NB_DIR))

import pandas as pd
from pipeline import (
    IE_CHECKPOINT_JSON,
    LLMNotAvailableError,
    LLM_REQUEST_DELAY_SECONDS,
    check_llm,
    get_llm_config,
    information_extraction_agent,
    load_cohort,
    load_ie_checkpoint,
    load_redaction_results,
    print_pipeline_banner,
    save_ie_checkpoint,
    save_ie_results,
    warn_if_slow_model,
)

print_pipeline_banner()
LLM_CONFIG = get_llm_config()
ok, model_info = check_llm(LLM_CONFIG)
if not ok:
    raise LLMNotAvailableError(model_info)
warn_if_slow_model(model_info, LLM_CONFIG.provider)
print(f"LLM ready — {LLM_CONFIG.method_prefix()}: {model_info}")

cohort_df = load_cohort()

# Stage 1.5 strips diagnosis information from the latest admission's note so IE can't just
# read the answer off the page. Required, not optional — run stage_01b_redact_notes.ipynb first.
redaction_df = load_redaction_results()
redacted_lookup = redaction_df.set_index("patient_id")["redacted_note"].to_dict()
cohort_df["clinical_note_original"] = cohort_df["clinical_note"]
cohort_df["clinical_note"] = cohort_df["patient_id"].astype(str).map(redacted_lookup)
missing = cohort_df["clinical_note"].isna()
if missing.any():
    raise ValueError(
        f"{missing.sum()} patient(s) missing redacted notes — run "
        "notebooks/stage_01b_redact_notes.ipynb for the full cohort first."
    )

print(f"Loaded cohort: {len(cohort_df)} patients (latest note + structured data each)")
print("Using diagnosis-redacted clinical notes (Stage 1.5)")


## Run Information Extraction

In [2]:
import time

checkpoint = load_ie_checkpoint()
rows = list(checkpoint.values())
done_hadm = set(checkpoint.keys())
print(f"Resuming: {len(done_hadm)} already done, {len(cohort_df) - len(done_hadm)} remaining")

for _, row in cohort_df.iterrows():
    hadm_id = int(row["hadm_id"])
    if hadm_id in done_hadm:
        continue

    history = row.get("admission_history") or []
    print(f"IE patient={row['patient_id']} hadm_id={hadm_id} ({len(history)} prior admissions)...")
    try:
        ctx = row.get("clinical_context_text") or ""
        extracted = information_extraction_agent(
            row["clinical_note"],
            config=LLM_CONFIG,
            admission_history=history,
            clinical_context_text=ctx,
        )
    except (TimeoutError, LLMNotAvailableError) as exc:
        save_ie_checkpoint(rows)
        raise type(exc)(
            f"{exc}\nCheckpoint saved ({len(rows)} patients) → {IE_CHECKPOINT_JSON}\n"
            "Re-run this cell to resume."
        ) from exc

    method = extracted.pop("_method", "unknown")
    record = {
        "patient_id": row["patient_id"],
        "admission_id": row["admission_id"],
        "subject_id": int(row["subject_id"]),
        "hadm_id": hadm_id,
        "note_type": row["note_type"],
        "n_prior_admissions": len(history),
        "admission_history": history,
        "clinical_context_text": ctx,
        "structured_vitals": row.get("structured_vitals") or [],
        "structured_labs": row.get("structured_labs") or [],
        "structured_reports": row.get("structured_reports") or [],
        "primary_icd_code": row["primary_icd_code"],
        "primary_dx_title": row["primary_dx_title"],
        "ground_truth_icd10": row["ground_truth_icd10"],
        "ground_truth_dx_titles": row["ground_truth_dx_titles"],
        "n_diagnoses": int(row["n_diagnoses"]),
        "extraction_method": method,
        "symptom_count": len(extracted.get("symptoms", [])),
        "diagnosis_count": len(extracted.get("diagnoses_mentioned", [])),
        "extracted": extracted,
    }
    rows.append(record)
    done_hadm.add(hadm_id)
    save_ie_checkpoint(rows)

    if LLM_REQUEST_DELAY_SECONDS > 0:
        time.sleep(LLM_REQUEST_DELAY_SECONDS)

results_df = pd.DataFrame(rows)
print(f"\nDone — {len(results_df)} patients (latest admission each)")
results_df[["patient_id", "admission_id", "n_prior_admissions", "extraction_method", "symptom_count"]]



Resuming: 15 already done, 0 remaining

Done — 15 patients (latest admission each)


,patient_id,admission_id,n_prior_admissions,extraction_method,symptom_count
0,10361982,24286431,1,openrouter_nlp:qwen/qwen-2.5-7b-instruct,3
1,10426859,29908281,2,openrouter_nlp:qwen/qwen-2.5-7b-instruct,5
2,10458324,21744342,1,openrouter_nlp:qwen/qwen-2.5-7b-instruct,3
3,11251337,29568708,2,openrouter_nlp:qwen/qwen-2.5-7b-instruct,4
4,11474876,29672491,1,openrouter_nlp:qwen/qwen-2.5-7b-instruct,4
5,11607177,23293838,4,openrouter_nlp:qwen/qwen-2.5-7b-instruct,5
6,12007928,23749816,2,openrouter_nlp:qwen/qwen-2.5-7b-instruct,7
7,13196707,21475988,1,openrouter_nlp:qwen/qwen-2.5-7b-instruct,6
8,13508515,21834271,1,openrouter_nlp:qwen/qwen-2.5-7b-instruct,4
9,13952483,23852410,2,openrouter_nlp:qwen/qwen-2.5-7b-instruct,13


## Save Results

In [ ]:
out = save_ie_results(results_df)
print(f"Saved → {out}")